In [2]:
import pandas as pd
import numpy as np



# Citire CSV-uri
clubs = pd.read_csv('../data/raw/clubs.csv')
players = pd.read_csv('../data/raw/players.csv')
valuations = pd.read_csv('../data/raw/player_valuations.csv')
appearances = pd.read_csv('../data/raw/appearances.csv')
club_games = pd.read_csv('../data/raw/club_games.csv')

# Creare coloana Age pentru jucatori
players['date_of_birth'] = pd.to_datetime(players['date_of_birth'])
players['age'] = np.floor(((pd.Timestamp.now() - players['date_of_birth']).dt.days / 365.25)).astype('Int64')

# Jucatori care merita considerati, jucatorii de peste 36 de ani nu mai au o cota de piata relevanta
players_active = players[(players['last_season'] >= 2025) & (players['age'] <= 36)]

# Top Jucatori
players_active.sort_values('market_value_in_eur', ascending = False)[['name', 'date_of_birth', 'age', 'market_value_in_eur']].head(10)

# Numarul de jucatori cu valori nule per coloana
players_active.isna().sum()

# jucatori care au cota de piata
players_with_value = players_active[
    (players_active['market_value_in_eur'].notna()) &
    (players_active['sub_position'].notna())
]
# grupare dupa pozitii
players_with_value.groupby('sub_position').agg(
    mean_market_value = ('market_value_in_eur', 'mean'),
    max_market_value = ('market_value_in_eur', 'max'),
    number_of_players = ('market_value_in_eur', 'count')
).sort_values('mean_market_value', ascending = False)

# adaugam o coloana in care avem valorile in milioane de euro
players_with_value['market_value_in_m'] = players['market_value_in_eur'] / 1_000_000
# grupare dupa varsta
players_with_value.groupby('age').agg(
    mean_market_value = ('market_value_in_m', 'mean'),
    number_of_players = ('market_value_in_m', 'count'),
    max_market_value = ('market_value_in_m', 'max')
).sort_values('age', ascending = True)


# Pasul 3 - Feature Engineering - Merge-uim clubs cu players
clubs_subset = clubs[['club_id', 'name', 'domestic_competition_id', 'squad_size']]
clubs_subset = clubs_subset.rename(columns = {'name': 'club_name'})
players_stats = players_with_value.merge(
    clubs_subset,
    left_on = 'current_club_id',
    right_on = 'club_id',
    how = 'left'
)
# luam meciurile din ultimii 3 ani
appearances['date'] = pd.to_datetime(appearances['date'])
recent_appearances = appearances[appearances['date'].dt.year >= 2023] # mai tarziu o sa revenim si o sa le numaram pe sezoane, nu pe ultimele 3 si atat
players_subset = players_stats[['player_id', 'name', 'country_of_citizenship', 'sub_position']]
player_stats = recent_appearances.groupby('player_id').agg(
    goals = ('goals', 'sum'),
    assists = ('assists', 'sum'),
    minutes_played = ('minutes_played', 'sum'),
)
player_stats = player_stats.reset_index()
players_stats = players_stats.merge(
    player_stats[['player_id', 'goals', 'assists', 'minutes_played']],
    on='player_id',
    how='left'
)
players_stats[['goals', 'assists', 'minutes_played', 'international_caps', 'international_goals']] = players_stats[['goals', 'assists', 'minutes_played', 'international_caps', 'international_goals']].fillna(0)

ml_data = players_stats[
    ['sub_position', 'foot', 'height_in_cm', 'age', 'country_of_citizenship', 'domestic_competition_id', 'squad_size',
     'goals', 'assists', 'minutes_played', 'international_caps', 'international_goals', 'market_value_in_m', 'player_id',
     'current_club_id'
    ]
]
ml_data = ml_data.dropna()


# creare statistica goals + assists

ml_data['contributions_per_90'] = (
    (ml_data['goals'] + ml_data['assists']) / ml_data['minutes_played'] * 90
).fillna(0)

# cream clean sheets pt fundasi si portari
app_games = pd.merge(recent_appearances,
                     club_games,
                     left_on = ['game_id', 'player_club_id'],
                     right_on = ['game_id', 'club_id'],
                     how = 'inner'
                     )
app_games['is_clean_sheet'] = ((app_games['opponent_goals'] == 0) & (app_games['minutes_played'] > 0)).astype(int)
clean_sheets_df = app_games.groupby('player_id').agg(
    clean_sheets = ('is_clean_sheet', 'sum')
).reset_index()
ml_data = ml_data.merge(
    clean_sheets_df,
    on = 'player_id',
    how = 'left'
).fillna(0)

defensive_positions = [
    'Goalkeeper', 'Centre-Back', 'Left-Back', 'Right-Back', 'Defensive Midfield'
]

ml_data.loc[~ml_data['sub_position'].isin(defensive_positions), 'clean_sheets'] = 0

# calculam o valoare pentru potentialul fiecarui jucator in fnuctie de varsta
# 29 de ani = apogeul, mai apoi valoarea incepe sa mai scada
# youth_minutes_score = minutes_played * max(0, 29 - age)
ml_data['youth_minutes_score'] = ml_data['minutes_played'] * (29 - ml_data['age']).clip(lower=0)

# calculam statistica goals + assists in functie de varsta
ml_data['contributions_score'] = (ml_data['goals'] + ml_data['assists']) * (29 - ml_data['age']).clip(lower = 0)

# calculam statistica clean sheets in functie de varsta pentru pozitiile mai defensive

ml_data['clean_sheets_score'] = (ml_data['clean_sheets']) * (29 - ml_data['age']).clip(lower = 0)

# calculam statisticile pentru ucl
ucl_stats = appearances[appearances['competition_id'] == 'CL'].groupby('player_id').agg(
    ucl_minutes_played = ('minutes_played', 'sum'),
    ucl_goals = ('goals', 'sum'),
    ucl_assists = ('assists', 'sum')
).reset_index()

ml_data = ml_data.merge(
    ucl_stats,
    on = 'player_id',
    how = 'left'
).fillna(0)


ml_data.to_csv('../data/processed/ml_data_ready.csv', index = False)


